# Simple examples - Extraction

This notebook is just a small reference on how to request OSM data via the ohsome-API:

- features for single timestamp
- collections for singe timestamp
- collections members for single timestamp
- contributions for time range

In [1]:
import os
from io import BytesIO

import geopandas as gpd
import httpx

In [2]:
OHSOME_API_URL = os.getenv("OHSOME_API_URL", "https://api.heigit.org/ohsome-api/v2-rc")
OHSOME_API_KEY = os.getenv(
    "OHSOME_API_KEY", ""
)  # insert your api key as env or default value here

## Features for single timestamp
A simple example to extract all bus stops in Heidelberg.

In [31]:
response = httpx.post(
    OHSOME_API_URL + "/extraction/features.parquet",
    json={
        "aoi": [8.6275437, 49.3727921, 8.7545042, 49.4385023],
        "filter": "highway=bus_stop and geometry:point",
        "time": "2026-06-01",
    },
    headers={"authorization": OHSOME_API_KEY},
)

gdf = gpd.read_parquet(
    BytesIO(response.content),
    to_pandas_kwargs={"maps_as_pydicts": "strict"},
)


gdf.explore(
    tiles="CartoDB positron",
    tooltip_kwds={"style": "max-width: 250px; white-space: normal;"},
)

## Collections for single Timestamp
We display the OSM relation "Buslinie 37" in Heidelberg.

In [32]:
response = httpx.post(
    OHSOME_API_URL + "/extraction/collections.parquet",
    json={
        "aoi": [8.6275437, 49.3727921, 8.7545042, 49.4385023],
        "filter": "type=route and route=bus and ref=37",
        "time": "latest",
        "clip": False,
    },
    headers={"authorization": OHSOME_API_KEY},
)

gdf = gpd.read_parquet(
    BytesIO(response.content),
    to_pandas_kwargs={"maps_as_pydicts": "strict"},
)

gdf.explore(
    tiles="CartoDB positron",
    tooltip_kwds={"style": "max-width: 250px; white-space: normal;"},
)

## Collections Members for single Timestamp
Here, we take a look at the members of the OSM relation "Buslinie 37".

We inspect the tags of the members to display the stop name. This name is part of the member and not of the parent relation.

In [33]:
response = httpx.post(
    OHSOME_API_URL + "/extraction/collections_members.parquet",
    json={
        "aoi": [8.6275437, 49.3727921, 8.7545042, 49.4385023],
        "filter": "type=route and route=bus and ref=37",
        "member_filter": "highway=bus_stop and geometry:point",
        "time": "latest",
        "clip": False,
    },
    headers={"authorization": OHSOME_API_KEY},
)

gdf = gpd.read_parquet(
    BytesIO(response.content),
    to_pandas_kwargs={"maps_as_pydicts": "strict"},
)

gdf["stop_name"] = gdf["tags"].apply(lambda x: x.get("name"))

gdf.explore(
    "stop_name",
    tiles="CartoDB positron",
    tooltip_kwds={"style": "max-width: 250px; white-space: normal;"},
)

## Contributions for Time Range
Here, we inspect all changes related to bus stops in Heidelberg between 2026-01 and 2026-06.

For each contribution we look at the OSM editor used. This properties comes from the changeset tags.

In [34]:
response = httpx.post(
    OHSOME_API_URL + "/extraction/contributions.parquet",
    json={
        "aoi": [8.6275437, 49.3727921, 8.7545042, 49.4385023],
        "filter": "highway=bus_stop and geometry:point",
        "time": {"start": "2026-01-01", "end": "2026-06-01"},
    },
    headers={"authorization": OHSOME_API_KEY},
)

gdf = gpd.read_parquet(
    BytesIO(response.content),
    to_pandas_kwargs={"maps_as_pydicts": "strict"},
)

gdf["editor"] = gdf["changeset_tags"].apply(lambda x: x.get("created_by"))

gdf.explore(
    "editor",
    tiles="CartoDB positron",
    tooltip_kwds={"style": "max-width: 250px; white-space: normal;"},
)